# Corpus Sampling, Weak Label Pool Construction & Lexicon Scoring Pipeline

This notebook implements the full data preparation pipeline I use to for getting the model training data together, both for the the auxiliary supervision models and the baseline. This pipeline is made up of 3 stages. First I draw 10 stratified random batches of 150 sentences from the full manifesto corpus for manual gold annotation across the three securitisation dimensions. I treat each manifesto as a distinct stratum & allocate sentences to batches proportional to document length to account for the uneven party and election-year coverage in the corpus without having to impose some kind of artificial uniform distribution.

Secondly I construct the weak-label pool by finding the set difference between the full corpus and the 1,500 gold-sampled sentences via exact text match. This ensures that no gold-annotated sentence enters the auxiliary supervision signal to keep a clean boundary between my two supervision sources.

Thirdly I apply a tier-weighted lexicon scoring procedure to every sentence in the weak pool, producing one aggregate weak score per securitisation dimension. Each labelling function operates independently over the tiered securitisation lexicon for its corresponding dimension, which I follow the weak supervision framework of Ratner et al. [1] for. At the end of this notebook I have a calibration analysis that evaluates how well the new lexicon-derived scores & my hand-annotated gold labels are aligned. I use that as a validity check on the weak supervision signal before model training.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
pipeline_config= {
    'num_batches': 10,
    'batch_size': 150,
    'random_seed_base': 42,
    'corpus_filename_prefix': 'manifesto_corpus',
    'lexicon_filename': 'securitisation_lexicon.xlsx',
    'gold_output_filename': 'manifesto_corpus_annotation_batches.xlsx',
    'weak_labels_output_filename': 'manifesto_corpus_weak_labels_scored_v6.xlsx',
    'gold_annotated_filename': 'manifesto_corpus_gold_with_splits.xlsx',
}

scoring_params= {
    'tier_weights': {
        'core': 3.0,
        'secondary': 1.5,
        'contextual': 0.75},
    'saturation_k': 1.2,# this is the saturation steepness, where higher values more aggressivley compress scores towards 1
    'use_length_normalisation': True,
    'length_norm_type': 'sqrt', # my options are None, 'log', or 'sqrt'
    'length_norm_floor': 4, # this is the min token count to prevent score inflation on really short fragments
    'lowercase': True,
    'remove_punctuation': True, # turning this on also strips apostrophes from input text & lexicon terms at load time
}

## Stage 1: Stratified Corpus Sampling
I'm producing 10 blocks of 150, stratified by overall party sentence cnt to deal with the disproportionate spread of sentences across the manifesto corpus. Once this is done I go away and label all 1500 of them to be used in all configs of the model (baseline, full aux & threat only).

In [ ]:
# I just find the 1st parquet file in this directory matching the corpus prefix
all_local_files= [f for f in os.listdir('.') if os.path.isfile(f)]
corpus_parquet_candidates= [f for f in all_local_files if f.startswith(pipeline_config['corpus_filename_prefix']) and f.endswith('.parquet')]
if not corpus_parquet_candidates:
    raise FileNotFoundError(f"hmm no parquet file starting with '{pipeline_config['corpus_filename_prefix']}' was actually present in this directory")

corpus_filename= corpus_parquet_candidates[0]
print(f"now loading full corpus from {corpus_filename}...")

try:
    full_corpus_df= pd.read_parquet(corpus_filename)
    print(f"loaded {len(full_corpus_df):,} rows, {full_corpus_df.shape[1]} columns")
except Exception as corpus_load_e:
    raise Exception(f"couldn't load corpus, error is {corpus_load_e}")

# I setup an int id used to track my sampled rows across batches
full_corpus_df['global_row_id']= full_corpus_df.index
display(full_corpus_df.head())

In [ ]:
# I run this 10x over to pull 10 batches of 150 labels each to hand-annotate (makes it manageable for me to do 150 at a time)
def draw_one_stratified_batch(pool_df, batch_size, random_seed):
    np.random.seed(random_seed)

    sentences_per_manifesto= pool_df['manifesto_id'].astype(str).value_counts().sort_index()
    proportion_of_corpus_per_manifesto= sentences_per_manifesto / len(pool_df)
    sentences_to_draw_per_manifesto= (proportion_of_corpus_per_manifesto * batch_size).round().astype(int)

    # I had to update this to correct rounding drift because of annoying bug, so now total allocation hits batch_size exactly
    rounding_correction_needed= batch_size - sentences_to_draw_per_manifesto.sum()
    if rounding_correction_needed != 0:
        manifestos_sorted_by_size= sentences_to_draw_per_manifesto.sort_values(ascending=False)
        for i in range(abs(rounding_correction_needed)):
            manifesto_to_adjust= manifestos_sorted_by_size.index[i % len(manifestos_sorted_by_size)]
            sentences_to_draw_per_manifesto[manifesto_to_adjust] += int(np.sign(rounding_correction_needed))

    sampled_sentence_frames= []
    for curr_manif_id, sentences_to_draw in sentences_to_draw_per_manifesto.items():
        manifesto_sentences= pool_df[pool_df['manifesto_id'].astype(str) == curr_manif_id]
        # this now has to guard against edge cases where allocation exceeds available rows in this stratum
        sentences_to_draw= min(sentences_to_draw, len(manifesto_sentences))
        sampled_sentence_frames.append(manifesto_sentences.sample(n=sentences_to_draw, replace=False, random_state=random_seed))

    # now I shuffle my assembled batch to get rid of this stratum ordering before export
    return pd.concat(sampled_sentence_frames).sample(frac=1, random_state=random_seed).reset_index(drop=True)

In [ ]:
batches_for_annotation= {}
row_ids_i_already_sampled= set()

# sampling pool shrinks after each batch so no sentence can appear twice
corpus_sentences_not_yet_sampled= full_corpus_df.copy()

for batch_number in range(pipeline_config['num_batches']):
    seed_for_this_batch= pipeline_config['random_seed_base'] + batch_number
    freshly_drawn_batch= draw_one_stratified_batch(corpus_sentences_not_yet_sampled, pipeline_config['batch_size'], seed_for_this_batch)

    row_ids_in_this_batch= set(freshly_drawn_batch['global_row_id'].tolist())
    row_ids_i_already_sampled.update(row_ids_in_this_batch)

    corpus_sentences_not_yet_sampled= corpus_sentences_not_yet_sampled[~corpus_sentences_not_yet_sampled['global_row_id'].isin(row_ids_in_this_batch)].copy()

    # here I strip all metadata myself (& any other future annotater) will sees sentence text & id.
    # That way there's a reduced risk of annotater bias from preconceptions about the party itself
    batch_stripped_for_annotation= freshly_drawn_batch[['sentence_id', 'sentence_text']].copy()
    batch_stripped_for_annotation['threat_score'], batch_stripped_for_annotation['urgency_score'], batch_stripped_for_annotation['justification_score'] = '', '', ''
    batches_for_annotation[f"Batch_{batch_number + 1}"]= batch_stripped_for_annotation

print(f"drawn {pipeline_config['num_batches']} batches, & there were {len(row_ids_i_already_sampled)} unique sentences sampled")

In [ ]:
with pd.ExcelWriter(pipeline_config['gold_output_filename'], engine='openpyxl') as gold_writer:
    for sheet_name, batch_df in batches_for_annotation.items():
        batch_df.to_excel(gold_writer, sheet_name=sheet_name, index=False)

print(f"great, gold batches saved to {pipeline_config['gold_output_filename']}")

## Stage 2: Weak Label Pool Construction

Now I construct the weak pool to be labelled, by 1st identifying gold sentences by exact text match and then excluding them from the full corpus to form the weak-label pool. I did this because text-match exclusion is more robust than index-based exclusion as its correct regardless of sentence ordering or corpus version differences.

In [ ]:
# now I collect exact texts of all sampled gold sentences for excluding them out of the weak one
gold_sentence_texts= set(pd.concat(batches_for_annotation.values())['sentence_text'].dropna().str.strip())
is_gold_sentence= full_corpus_df['sentence_text'].str.strip().isin(gold_sentence_texts)
weak_pool_df= (full_corpus_df[~is_gold_sentence][['sentence_id', 'sentence_text']].drop_duplicates(subset='sentence_text').reset_index(drop=True))

n_leaked= weak_pool_df['sentence_text'].str.strip().isin(gold_sentence_texts).sum() # I had a bug w/ this before so now I'm putting in guardrails
print(f"overall I excluded {is_gold_sentence.sum()} gold rows")
print(f"and weak pool size is now {len(weak_pool_df):,} manifesto sentences")
print(f"sanity checking gold sentences in weak pool, there's (should be 0): {n_leaked}")

## Stage 3: Lexicon Loading & Scoring

Now for the weak scoring. I score each sentence in the weak pool using a tier-weighted lexicon matching procedure (you can read about that in my paper too). The pipeline operates in three steps. a) a greedy longest-match scan that aggregates weighted term hits per tier b) an (optional, through above hyperparams as a toggle) length normalisation that divides the weighted sum by the square root of token count and c) a saturating transformation of the form $1 - e^{-kS}$ that models diminishing marginal contribution from additional lexicon matches, altogether producing one aggregate score per securitisation dimension.

The greedy longest-match scan fixes multi-word entry collisions by always preferring the longest matching phrase at each position to prevent double-counting. I also apply punctuation stripping to both the input text & lexicon terms at load time, which handles apostrophe-containing vocabulary consistently across the full lexicon.

In [ ]:
# this loads in the 3 different lexicons I constructed in previous notebooks, pulling each dimension out of its corresponding excel tab
def loadin_lexicons():
    lexicon_path= pipeline_config['lexicon_filename']
    if not os.path.exists(lexicon_path):
        raise FileNotFoundError(f"hmm couldn't find it at {lexicon_path}, x2 check it's there.")

    try:
        print(f"pulling lexicons from {lexicon_path} now...")
        raw_lex_sheets= pd.read_excel(lexicon_path, sheet_name=['Threat Lexicon', 'Urgency Lexicon', 'Justification Lexicon'])

        def build_tiered_lexicon(df):
            df= df.iloc[:, :2].dropna()
            df.columns= ['term', 'tier']
            df['term']= df['term'].str.lower()
            df['tier']= df['tier'].str.lower()
            # punctuation stripping here handles apostrophes in terms like "putin's"
            # so they match the same stripping applied to input text in preprocess()
            if scoring_params['remove_punctuation']:
                df['term']= df['term'].str.replace(r'[^\w\s]', '', regex=True)
            return {
                'core': set(df[df['tier'] == 'core']['term']),
                'secondary': set(df[df['tier'] == 'secondary']['term']),
                'contextual': set(df[df['tier'] == 'contextual']['term']),
            }
        
        # getting them from their tabs
        loaded_lexicons= {
            'threat': build_tiered_lexicon(raw_lex_sheets['Threat Lexicon']),
            'urgency': build_tiered_lexicon(raw_lex_sheets['Urgency Lexicon']),
            'justification': build_tiered_lexicon(raw_lex_sheets['Justification Lexicon']),}
        for dim, lex in loaded_lexicons.items():
            n_terms= sum(len(v) for v in lex.values())
            print(f"{dim}: {n_terms} terms (core={len(lex['core'])}, secondary={len(lex['secondary'])}, contextual={len(lex['contextual'])})")

        return loaded_lexicons

    except Exception as lexicon_load_e:
        raise Exception(f"hmm that's a problem, couldn't load lexicons. Error is {lexicon_load_e}")

lexicons= loadin_lexicons()

In [ ]:
def preprocess_for_aggregation_scoring(text):
    text= str(text)
    if scoring_params['lowercase']:
        text= text.lower()
    if scoring_params['remove_punctuation']:
        text= re.sub(r'[^\w\s]', '', text)
    return text.split()


# this is my greedy left-to-right scan I mention in my report
# It always tries longest match first (tri over bi over uni gram)
# so multi-word entries take priority over constituent unigrams & repeated occurrences are counted
def priority_match_toks(manif_toks, lexicon_dict):
    term_to_tier= {}
    for tier, terms in lexicon_dict.items(): # i got tier-by-tier for each securitisation dim
        for term in terms:
            term_to_tier[term]= tier
    # i.e i got no matches
    if not term_to_tier:
        return {'core': 0, 'secondary': 0, 'contextual': 0}

    max_possible_n= max(len(term.split()) for term in term_to_tier)
    overall_lex_hits= {'core': 0, 'secondary': 0, 'contextual': 0} # cumulative hit counter, which the saturation function accounts for

    running_cnt= 0 # check how many matches for this particular dim
    while running_cnt < len(manif_toks):
        matched= False
        for each_n in range(min(max_possible_n, len(manif_toks) - running_cnt), 0, -1):
            manif_phrase= ' '.join(manif_toks[running_cnt:running_cnt + each_n])
            if manif_phrase in term_to_tier:
                overall_lex_hits[term_to_tier[manif_phrase]] += 1 # i.e. got a hit for this specific tier & dim
                running_cnt += each_n
                matched= True
                break
        if not matched: # just proceed w/out update
            running_cnt += 1
    return overall_lex_hits

# this takes into account the length of the manifesto sentence (toggleable hyperparm above so I can experiment w/ effects of this)
def normalise_manif_sent_wrt_len(score, token_count):
    if not scoring_params['use_length_normalisation']:
        return score
    # I added this floor to stop score inflation on v short fragments (like titles or headers alone)
    effective_length= max(token_count, scoring_params['length_norm_floor'])
    if scoring_params['length_norm_type'] == 'log': # (another hyperparam I can toggle above, for ablating to figure out what gives me the best results)
        return score / np.log1p(effective_length)
    elif scoring_params['length_norm_type'] == 'sqrt':
        return score / np.sqrt(effective_length)
    return score

# I chose this 1 - exp(-kS) function because it models diminishing marginal contribution from any extra lexicon hits
# I upskilled using this website: https://www.indaru.com/mmm-saturation-curves/#:~:text=Lastly%2C%20this%20function%20is%20most,Logistic%20Function:
def saturate_scoring_func(score):
    return 1 - np.exp(-scoring_params['saturation_k'] * score)

# now the actual main funtion that calcualtes what the aggregate score is going to be for that sentence-dim pair
def calc_aggregate_score(text, dimension):
    sent_toks= preprocess_for_aggregation_scoring(text)
    tier_specific_weights= scoring_params['tier_weights']
    hits= priority_match_toks(sent_toks, lexicons[dimension])
    # so now I find produce my final weighted sum score that takes into account all the factors I've mentioned above
    weighted_sum= (hits['core'] * tier_specific_weights['core'] + hits['secondary'] * tier_specific_weights['secondary'] + hits['contextual'] * tier_specific_weights['contextual'])
    return saturate_scoring_func(normalise_manif_sent_wrt_len(weighted_sum, len(sent_toks)))

In [ ]:
# so i end up now with one aggregate score per dimension, which are my version of the labelling functions Ratner et al. [1] comes up with
for dim in ['threat', 'urgency', 'justification']:
    weak_pool_df[f'{dim}_weak_score']= weak_pool_df['sentence_text'].apply(lambda x: calc_aggregate_score(x, dim))

print(f"good, scoring all done. That gives me {len(weak_pool_df):,} rows & {len(weak_pool_df.columns)} columns")
display(weak_pool_df.head())

## Lexicon Coverage Report

I'm choosing to report per-dimension coverage before saving to confirm that the lexicons do actually provide a meaningful signal spread across the corpus. Low coverage would suggest that pseudo-labels are near-zero for the majority of sentence which would degrade the usefulness of the auxiliary supervision signal I'm giving my variant models.

In [ ]:
print(f"lexicon coverage for the weak pool ({len(weak_pool_df):,} sentences)")

for dim in ['threat', 'urgency', 'justification']:
    score_col= f'{dim}_weak_score'
    sentences_with_hits= (weak_pool_df[score_col] > 0).sum()
    coverage_pct= 100.0 * sentences_with_hits / len(weak_pool_df)
    nonzero_scores= weak_pool_df.loc[weak_pool_df[score_col] > 0, score_col]
    mean_nonzero= nonzero_scores.mean() if len(nonzero_scores) > 0 else 0.0
    print(f"I get {dim}: {sentences_with_hits} sentences with hits ({coverage_pct:.1f}%) with mean non-zero score {mean_nonzero:.4f}")

In [ ]:
weak_pool_df.to_excel(pipeline_config['weak_labels_output_filename'], index=False)
print(f"alignment check is done and weak labels are saved at {pipeline_config['weak_labels_output_filename']} (consisting of {len(weak_pool_df):,} rows)")

## Gold Calibration Analysis

Penultimately I evaluate alignment between the new lexicon-derived weak scores and hand-annotated gold labels by applying the same scoring pipeline to the gold sentence set. I don't need nor expect these weak labells to perfectly replicate human judgements, as the objective is just to confirm a directionally meaningful correlation that justifies me using them as a noisy (as permitted & expected by Ratner's methodology) auxiliary supervision signal. I'm reporting Pearson and Spearman correlations & MAE per dimension, followed by a per-tier hit breakdown and distributional comparison.

### Metrics I calculate here

I chose to use a range of different metrics based on my wider readings of publications in the field I was investigating. I've determined that these seem to be the standard ones, as I've seen them all pop up in context before.

- **Pearson r**, which I use tio measure linear correlation between my weak and gold scores i.e. whether the two scales move together proportionally. It's useful for seeing if the lexicon tracks absolute intensity.

- **Spearman p** which measures rank-order correlation and is the more important one here since the model's downstream use is ordering sentences by securitisation intensity, not hitting exact values.

- **MAE** which gives the average absolute gap between weak and gold scores in interpretable [0,1] units, and is a pretty good direct read on typical disagreement.

- **% coverage** I use to report what % of gold sentences got a non-zero weak score. Low coverage would mean most of my gold sentences are scored 0 by the lexicon regardless of their actual label, which would make the weak signal useless as auxiliary supervision.

In [ ]:
# reading it back in (assuming I've gone and annoated it, it won't work unannoated of course)
try:
    gold_annotated_df= pd.read_excel(pipeline_config['gold_annotated_filename'])
    print(f"loaded gold annotated set: {len(gold_annotated_df)} rows")
except Exception as gold_load_e:
    raise Exception(f"hmm, couldn't load gold annotated file: {gold_load_e}")

# now I apply the same weak scoring pipeline to gold sentences
for dim in ['threat', 'urgency', 'justification']:
    gold_annotated_df[f'{dim}_weak_score']= gold_annotated_df['sentence_text'].apply(lambda x: calc_aggregate_score(x, dim))
gold_annotated_df.head()

In [ ]:
print('testing weak vs gold alignment (for reporting in my final writeup)')

wvg_calibration_results= []

for dim in ['threat', 'urgency', 'justification']:
    gold_scores= gold_annotated_df[f'{dim}_score']
    weak_scores= gold_annotated_df[f'{dim}_weak_score']

    pearson_r, pearson_p= pearsonr(gold_scores, weak_scores)
    spearman_rho, spearman_p= spearmanr(gold_scores, weak_scores)
    mae= mean_absolute_error(gold_scores, weak_scores)
    gold_coverage_pct= 100.0 * (weak_scores > 0).sum() / len(weak_scores)

    # now I store all the various calibration metrics I logged in this for dispalying as a neat df
    wvg_calibration_results.append({
        'dimension': dim, 'pearson_r': pearson_r, 'pearson_p': pearson_p,
        'spearman_rho': spearman_rho, 'spearman_p': spearman_p,
        'mae': mae, 'gold_coverage_pct': gold_coverage_pct,})
    print(f"for {dim}, I get: pearson={pearson_r:.4f} (p={pearson_p:.2e}), spearman={spearman_rho:.4f} (p={spearman_p:.2e}), MAE={mae:.4f} & coverage={gold_coverage_pct:.1f}%")

display(pd.DataFrame(wvg_calibration_results))

In [ ]:
# now for the per-tier hit breakdown
# a dimension overly dominated by contextual hits has weaker signal than one with 
# many core hits, which explains the variance in alignment across dimensions I'm seeing
print('\nper-tier hit breakdown for the gold set')

for dim in ['threat', 'urgency', 'justification']:
    core_total= secondary_total= contextual_total= sentences_with_any_hit= 0
    # I go row by row along the new  hand-annotated (outside this notebook) gold set created in this file 
    for _, gold_row in gold_annotated_df.iterrows(): # (skip cols, irrelevant to me)
        sent_toks= preprocess_for_aggregation_scoring(gold_row['sentence_text'])
        hits_in_this_row= priority_match_toks(sent_toks, lexicons[dim])
        if any(hits_in_this_row[hit] > 0 for hit in hits_in_this_row):
            sentences_with_any_hit += 1

        core_total += hits_in_this_row['core']
        secondary_total += hits_in_this_row['secondary']
        contextual_total += hits_in_this_row['contextual']

    total_hits= core_total + secondary_total + contextual_total
    overall_hit_pct= 100 * sentences_with_any_hit / len(gold_annotated_df)
    print(f"for {dim} I found {sentences_with_any_hit}/{len(gold_annotated_df)} sentences with hits ({overall_hit_pct:.1f}%), sitting at a total of {total_hits} (of those, core={core_total}, secondary={secondary_total}, contextual={contextual_total})")

## Score Distribution Comparison
Finally I made this visual to visualise how the weak label distribution scoring correlates w/ my hand-annotated gold ones.

In [ ]:
for dim in ['threat', 'urgency', 'justification']:
    plt.figure()
    plt.hist(gold_annotated_df[f'{dim}_score'], bins=10, alpha=0.6, label='Gold')
    plt.hist(gold_annotated_df[f'{dim}_weak_score'], bins=20, alpha=0.6, label='Weak')
    plt.xlabel('score')
    plt.ylabel('freq.')
    plt.title(f'score distribution for {dim.capitalize()}')
    plt.legend()
    plt.show()

## References

[1] A. Ratner, S. H. Bach, H. Ehrenberg, J. Fries, S. Wu, and C. Ré, "Snorkel," *Proceedings of the VLDB Endowment*, vol. 11, no. 3, pp. 269–282, Nov. 2017, doi: https://doi.org/10.14778/3157794.3157797.